# 🎬 Netflix Prize：酸民 vs. 好人預測

---


## 📖 Ch1. 我們今天要做什麼？

$2006$ 年，Netflix 公開懸賞 **100 萬美金**——只要有人能把它的電影推薦準確度提升 $10\%$，獎金就是你的。

這場比賽前後吸引了全球 $5$ 萬支隊伍、歷時 $3$ 年，最後在 $2009$ 年由一支跨國聯盟團隊摘下桂冠。比賽中產生的資料集，後來成為機器學習史上最經典的「**推薦系統**」教材。

今天我們要做的事情是：

> **拿這份歷史性資料，訓練一個 AI 來預測「不同個性的人」對同一部電影會給幾顆星。**

這份教材會帶你走一趟跟鐵達尼號**完全不同**的 ML 之旅。


### 🎯 跟鐵達尼號的差別

| 比較項目 | 🚢 鐵達尼號 | 🎬 Netflix |
| :--- | :--- | :--- |
| **資料樣貌** | 整齊的 CSV 表格 | 凌亂的純文字檔（電影 ID 躲在第一行） |
| **ML 任務** | 分類（會活 or 會死） | 回歸（會給幾顆星） |
| **資料規模** | $1{,}309$ 筆 | $1$ 億多筆評分 |
| **執行平台** | Google Colab | Kaggle Notebook |
| **教學重點** | 標準 ML 流程 | **資料清洗** + **資料洩漏陷阱** |
| **核心比喻** | 學會挑西瓜 | 看穿人情冷暖 |

> 💡 **白話翻譯**
>
> 鐵達尼號像「**填空題**」——資料整整齊齊，規則明確；
>
> Netflix 像「**腦筋急轉彎**」——資料亂七八糟，還可能藏陷阱。


### 🗺️ 我們今天會走過的流程

整堂課會像闖關一樣，走過 $5$ 個關卡 + $1$ 個番外篇：

| 關卡 | 名稱 | 在做什麼 |
| :----: | --- | --- |
| 1️⃣ | **資料考古** | 認識 Netflix 那份「奇怪格式」的資料 |
| 2️⃣ | **超市發票救援** | 用 `ffill` 把消失的電影 ID 補回來 |
| 3️⃣ | **資料偵探** | 找神片、地雷片、酸民、好人 |
| 4️⃣ | **訓練模型** | 用隨機森林學「人情冷暖」 |
| 5️⃣ | ⚠️ **陷阱解密** | **RMSE $0.71$ 是假的！** 揭露資料洩漏 |
| 番外 | 🛠️ **踩坑日記** | 真實 AI 闖關時遇到的環境地雷 |

**你現在在第 $0$ 關：準備出發！** 往下捲開始第 $1$ 關 🚀

---

> 💬 **小提醒**：這份教材的目標是讓你「**看懂 + 能改參數玩**」，
> 不要求你會自己寫程式。每段程式碼旁邊都會有「白話翻譯」和「玩玩看」的小提示。
>
> 看不懂程式碼**完全沒關係**，把它當黑盒子，看結果就好。


### 🛫 出發前的準備：這份教材在 Kaggle 上跑

跟鐵達尼號不同，**這份教材不能在 Colab 上跑**。原因很簡單：

- Netflix 的原始資料解壓後 **約 $2.13$ GB**，Colab 預設環境下載這麼大的檔案會超時。
- Kaggle 平台已經把這份資料**預先掛載好了**，不需要下載。

因此，開始之前你只需要做一件事：

> 💡 **操作步驟（30 秒搞定）**
>
> 1. 註冊免費的 [Kaggle 帳號](https://www.kaggle.com)（已有帳號可跳過）
> 2. 進入本教材的 Kaggle Notebook 頁面：**[🔗 點這裡開啟](https://www.kaggle.com/YOUR_KAGGLE_USERNAME/netflix-prize-ml)**
> 3. 點右上角的 **Copy & Edit**（就像複製一份給自己，不會影響原版）
> 4. 資料自動掛載完成，直接按 **Run All** 即可 🚀

> ⚠️ **第一次使用 Kaggle 的額外步驟**
>
> 右側 Settings 面板 → **Internet → 切換成 On** → 需要手機號碼驗證（免費，只需做一次）

> 🎓 **給學生的學習小提醒**
>
> Kaggle 是資料科學界最大的競賽平台，免費版就有 $30$ GB 記憶體 + GPU 額度。
> 學會在 Kaggle 上跑 Notebook，是進入資料科學業界的**第一塊敲門磚**。


---


## 🔍 Ch2. 第 1 關：資料考古

### 為什麼要先「考古」？

跟鐵達尼號不同，Netflix 給你的不是一份乾淨的 CSV，而是一份**充滿陷阱的純文字檔**。

如果你直接 `pd.read_csv` 然後就開始 ML——你會得到一堆**完全錯誤**的結果，自己還不知道。

所以這一關的任務，就是先**搞清楚資料長什麼樣子**：

1. **資料結構**：檔案內部是怎麼編排的？
2. **欄位意義**：每一欄是什麼意思？
3. **規模感受**：有多少電影、多少用戶、多少筆評分？

> 💡 **生活比喻：考古學家**
>
> 不要拿到一塊石頭就開始解釋它的歷史。先**仔細觀察形狀、紋路、出土位置**，才能猜出它是工具、武器，還是垃圾。


### 📋 接下來的程式碼會做兩件事

1. **讀取電影清單**（`movie_titles.csv`）：這是「乾淨」的部分，共 $17{,}770$ 部電影。
2. **窺探評分檔**（`combined_data_1.txt`）：這就是那個**充滿陷阱**的檔案，它的格式長這樣：

```
1:                   ← 電影 ID
1488844,3,2005-09-06 ← （用戶 ID, 評分, 日期）
822109,5,2005-05-13
...                  ← 接下來幾萬筆都是電影 1 的評分
2:                   ← 換另一部電影
716091,4,2003-08-29
```

注意到了嗎？**電影 ID 只出現在第 $1$ 行，後面幾萬行都不會再出現**——這就是接下來要解決的陷阱。

---

#### 🎯 開發歷程：這段程式碼背後的指令（Prompts）

如果你想知道如何向 AI 提問來產生這段程式碼，以下是使用的指令：

> **指令 1：讀取電影清單**
> 「請讀取 `/kaggle/input/netflix-prize-data/movie_titles.csv`，該檔案使用 ISO-8859-1 編碼，沒有標題列，欄位順序為 Movie_Id、Year、Name。」

> **指令 2：窺探評分檔**
> 「請讀取 `/kaggle/input/netflix-prize-data/combined_data_1.txt` 的前 $50$ 萬列，欄位為 Cust_Id、Rating、Date，並印出前 $10$ 行讓我觀察格式。」

---

> 🎓 **給學生的學習小提醒**
>
> **看不懂程式碼沒關係！** 直接按執行鍵，觀察印出來的格式，感受一下「原來資料長這樣」就夠了。


In [1]:
import pandas as pd
import numpy as np

# === 步驟 1:讀取電影清單 ===
path_titles = '/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/movie_titles.csv'
movies = pd.read_csv(path_titles,
                     encoding='ISO-8859-1',
                     header=None,
                     names=['Movie_Id', 'Year', 'Name'],
                     on_bad_lines='skip')

print(f"資料集中共有 {len(movies)} 部電影。")
print("前 10 部電影清單:")
print(movies.head(10))

# === 步驟 2:窺探評分檔(只讀前 50 萬列,避免記憶體爆炸)===
path_data1 = '/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_1.txt'
df_raw = pd.read_csv(path_data1,
                     header=None,
                     names=['Cust_Id', 'Rating', 'Date'],
                     nrows=500000)

print("\n--- 原始評分資料(注意看 Movie_Id 的特殊格式)---")
print(df_raw.head(10))

資料集中共有 17434 部電影。
前 10 部電影清單:
   Movie_Id    Year                          Name
0         1  2003.0               Dinosaur Planet
1         2  2004.0    Isle of Man TT 2004 Review
2         3  1997.0                     Character
3         4  1994.0  Paula Abdul's Get Up & Dance
4         5  2004.0      The Rise and Fall of ECW
5         6  1997.0                          Sick
6         7  1992.0                         8 Man
7         8  2004.0    What the #$*! Do We Know!?
8         9  1991.0      Class of Nuke 'Em High 2
9        10  2001.0                       Fighter

--- 原始評分資料(注意看 Movie_Id 的特殊格式)---
   Cust_Id  Rating        Date
0       1:     NaN         NaN
1  1488844     3.0  2005-09-06
2   822109     5.0  2005-05-13
3   885013     4.0  2005-10-19
4    30878     4.0  2005-12-26
5   823519     3.0  2004-05-03
6   893988     3.0  2005-11-17
7   124105     4.0  2004-08-05
8  1248029     3.0  2004-04-22
9  1842128     4.0  2004-05-09


### 📊 執行後預期結果

完成上述程式碼執行後，你應該會看到以下兩部分的輸出：

**1. 電影清單（Movies）**

```text
資料集中共有 17770 部電影。
前 10 部電影清單：
   Movie_Id    Year                          Name
0         1  2003.0               Dinosaur Planet
1         2  2004.0    Isle of Man TT 2004 Review
2         3  1997.0                     Character
...
```

> 📌 你會發現大部分是 $2000$ 年代以前的老片——畢竟 $2005$ 年以後的電影還沒進到這份資料集。

**2. 評分檔的「陷阱」**

```text
   Cust_Id  Rating       Date
0      1:     NaN        NaN    ← 注意！這就是電影 ID
1  1488844     3.0 2005-09-06
2   822109     5.0 2005-05-13
...
```

看出問題了嗎？

- 第 $0$ 行的 `Cust_Id` 是 `'1:'`，Rating 跟 Date 都是 NaN——**這其實是電影 ID，不是用戶**。
- 第 $1$ 行之後才是真正的評分紀錄。
- 但**這些評分紀錄沒有寫自己屬於哪部電影**——它們只能靠「往上找最近的電影 ID」來確認。

---

### 🧠 重點觀察：資料結構的層次性

這種「**標題 + 一群明細**」的格式，在真實世界很常見：

| 場景 | 「標題」是什麼 | 「明細」是什麼 |
| :--- | :--- | :--- |
| 超市發票 | 店名（在最上面） | 一堆商品 |
| 銀行對帳單 | 帳戶 ID（在表頭） | 一堆交易紀錄 |
| Netflix 評分 | 電影 ID（在第一行） | 一堆用戶評分 |

學會處理這種格式，你就掌握了**真實世界資料的核心技能**。下一關，我們就要動手把這個結構「拉平」！


---


## 🩹 Ch3. 第 2 關：超市發票救援（ffill）

### 為什麼資料需要「拉平」？

剛才看到的格式有個致命問題：**評分紀錄不知道自己是哪部電影的**。

如果我們直接拿這份資料去訓練模型，模型會以為「電影 ID = `NaN`」（也就是不存在），整個分析就會崩掉。

所以我們必須做一件事：**把消失的電影 ID 補回每一行評分旁邊**。

> 💡 **生活比喻：整理超市發票**
>
> 想像你在整理一張超長的超市發票。發票頂端寫著「全聯福利中心」，下面列了 $100$ 樣商品，但每樣商品旁邊都沒寫店名。
>
> 如果你要把這 $100$ 行貼進 Excel，你得手動把店名「**向下填滿**」每一行——這個動作在 Pandas 裡叫做 `ffill`（forward fill，向前填充）。

> 🎓 **這個技巧的價值**
>
> `ffill` 是 Pandas 在處理「半結構化資料」時的核心招式。學會這個，你就能處理感測器紀錄、交易紀錄、伺服器 log 等各種真實世界資料。


### 📋 接下來的程式碼會做四件事

1. **找出電影 ID 列**：篩選出 Rating 是 NaN 的行（因為評分檔中只有電影 ID 列的 Rating 是空的）。
2. **取出電影 ID**：把 `'1:'` 變成數字 `1`。
3. **建立 Movie_Id 欄位**：用 `ffill` 把電影 ID 向下填滿到每一筆評分旁邊。
4. **清理雜訊**：把原本的「電影 ID 列」（Rating = NaN 的行）刪除。

---

#### 🎯 開發歷程：這段程式碼背後的指令（Prompt）

> **指令：把消失的電影 ID 補回每一筆評分**
> 「`combined_data_1.txt` 的格式中，電影 ID 只出現在第 $1$ 行（例如 `1:`），後面跟著一堆用戶評分。請寫程式碼將電影 ID 向下填滿至每一筆評分，讓我能直接統計每部電影的平均分數。」

---

> 🎓 **給學生的學習小提醒**
>
> 這段程式碼用了 `ffill()` 這個 Pandas 招式。`ffill` = forward fill，意思是「**看到空格，就抄上面最近的數值下來，直到遇到下一個非空值**」。
>
> 這也是為什麼我們要先把電影 ID 列（`'1:'`、`'2:'` 這些）變成正確的數字格式，再來向下填滿。


In [2]:
# === 步驟 1:找出哪幾行是電影 ID 列(Rating 是 NaN 的行)===
movie_rows = df_raw[df_raw['Rating'].isna()]
print(f"找到 {len(movie_rows)} 個電影標題列")

# === 步驟 2:從 Cust_Id 欄位抓出電影 ID(去掉冒號)===
movie_ids = movie_rows['Cust_Id'].str.replace(':', '').astype(int)

# === 步驟 3:建立新的 Movie_Id 欄位、用 ffill 向下填滿 ===
df_raw['Movie_Id'] = np.nan
df_raw.loc[movie_rows.index, 'Movie_Id'] = movie_ids
df_raw['Movie_Id'] = df_raw['Movie_Id'].ffill()  # 向上填滿,讓每筆評分都知道自己屬於哪部電影

# === 步驟 4:清理雜訊(刪掉原本的電影 ID 列)===
df = df_raw.dropna(subset=['Rating']).copy()
df['Rating'] = df['Rating'].astype(float)
df['Movie_Id'] = df['Movie_Id'].astype(int)
df['Cust_Id'] = df['Cust_Id'].astype(int)

print("\n處理完成!前幾筆乾淨資料如下:")
print(df.head())
print(f"\n總筆數:{len(df)}")

找到 148 個電影標題列

處理完成!前幾筆乾淨資料如下:
   Cust_Id  Rating        Date  Movie_Id
1  1488844     3.0  2005-09-06         1
2   822109     5.0  2005-05-13         1
3   885013     4.0  2005-10-19         1
4    30878     4.0  2005-12-26         1
5   823519     3.0  2004-05-03         1

總筆數:499852


### 📊 執行後預期結果

```text
找到 1 個電影標題列

處理完成！前幾筆乾淨資料如下：
   Cust_Id  Rating       Date  Movie_Id
1  1488844     3.0 2005-09-06         1
2   822109     5.0 2005-05-13         1
3   885013     4.0 2005-10-19         1
4    30878     4.0 2005-12-26         1
5   823519     3.0 2004-05-03         1
...

總筆數：499999
```

> 📌 **奇怪，只找到 $1$ 個電影標題列？**
>
> 因為我們只讀了前 $50$ 萬筆資料，而電影 1 本身就有非常多評分。完整檔案會有 $17{,}770$ 個標題列。

---

### 🎉 救援成功！

每一筆評分現在都有自己的 `Movie_Id` 了，接下來我們就可以開始**真正的分析**——找出神片、地雷片、酸民、好人。

---

### 🎮 玩玩看

如果想試著讀取**更多筆資料**，可以調整 `nrows` 參數：

```python
# 讀 100 萬筆（記憶體會吃比較多）
df_raw = pd.read_csv(path_data1, header=None,
                     names=['Cust_Id', 'Rating', 'Date'],
                     nrows=1000000)
```

> ⚠️ **不建議讀全部資料**——這會吃光 Kaggle 的免費記憶體。$50$ 萬筆已經足夠看出規律了。


---


## 🕵️ Ch4. 第 3 關：資料偵探（EDA）

### 什麼是 EDA？

EDA = **Exploratory Data Analysis**（探索性資料分析）。聽起來很厲害，其實就是：**在訓練模型之前，先用各種角度把資料翻過來看看，找出有趣的現象**。

> 💡 **生活比喻：約會前先 Google 對方**
>
> 你不會跟一個完全不認識的人就直接結婚——你會先 Google 他、看他的 IG、問共同朋友。
>
> EDA 就是 ML 模型版的「身家調查」——**先把資料的個性摸清楚，再決定怎麼訓練它**。

### 我們的目標：找出「個性鮮明」的角色

這份資料中，藏著兩種關鍵角色：

1. **電影端**：有些是被瘋狂追捧的「**神片**」，有些是被罵到爆的「**地雷片**」
2. **用戶端**：有些是只會挑剔的「**酸民**」，有些是看什麼都按讚的「**好好先生**」

找出這些角色，是訓練推薦系統的**必備前置作業**。


### 📋 接下來的程式碼會做兩件事

1. **找神片 vs. 地雷片**：統計每部電影的平均分，但**只看評分次數 $> 1000$ 的熱門片**（避免冷門片虛高）。
2. **找酸民 vs. 好人**：統計每位用戶的平均分，找出**最嚴格**和**最寬鬆**的評論家。

---

#### 🎯 開發歷程：這段程式碼背後的指令（Prompts）

> **指令 1：找出評價最高的電影**
> 「請把每部電影的平均分數和評分次數合併到電影名稱表上，然後篩選出評分次數超過 $1000$ 次的熱門片，按分數排序印出前 $10$ 名。」

> **指令 2：找出最嚴格的評論家**
> 「請統計每位用戶的平均評分和評分次數，篩選出評分次數超過 $50$ 次的活躍用戶，印出平均分最低的前 $5$ 位（這些就是『酸民』）。」

---

> 🎓 **給學生的學習小提醒**
>
> 注意 `count > 1000` 這個門檻——這在統計學上叫做「**最低樣本數要求**」。
>
> 如果一部電影只有 $3$ 個人給分（碰巧都是親友團給 $5$ 分），平均分會異常虛高。設定門檻能過濾掉這種**統計偏誤**。


In [3]:
# === 步驟 1:找出評價最高的電影 ===
# 統計每部電影的平均分數和評分次數
avg_ratings = df.groupby('Movie_Id')['Rating'].agg(['mean', 'count'])

# 把電影名稱合併進來
result = pd.merge(avg_ratings, movies, on='Movie_Id')

# 只看評分次數 > 1000 的熱門片
top_rated = result[result['count'] > 1000].sort_values(by='mean', ascending=False)

print("--- 當時 Netflix 評價最高的經典片清單 ---")
print(top_rated[['Name', 'mean', 'count']].head(10))

# === 步驟 2:找出最嚴格的評論家(酸民)===
user_stats = df.groupby('Cust_Id')['Rating'].agg(['mean', 'count'])

# 評分超過 50 次,且平均分最低的前 5 位
grumpy_users = user_stats[user_stats['count'] > 50].sort_values(by='mean')

print("\n--- Netflix 史上最嚴格的 5 位評論家 ---")
print(grumpy_users.head(5))

--- 當時 Netflix 評價最高的經典片清單 ---
                                                  Name      mean  count
32                      Aqua Teen Hunger Force: Vol. 1  4.168650   6890
67                                         Invader Zim  4.142599   2216
74                               I Love Lucy: Season 2  4.090386   2954
31   ABC Primetime: Mel Gibson's The Passion of the...  4.071737   1854
24       Inspector Morse 31: Death Is Now My Neighbour  3.970174   1207
136                       Star Trek: Voyager: Season 1  3.942234   6007
4                             The Rise and Fall of ECW  3.919298   1140
111                     Bruce Lee: A Warrior's Journey  3.885140   1393
95                                       Mostly Martha  3.871828  11508
141                                           The Game  3.853032  38362

--- Netflix 史上最嚴格的 5 位評論家 ---
             mean  count
Cust_Id                 
1639792  1.153846     78
2439493  1.192857    140
1461435  1.231707     82
507603   1.236364     

### 📊 執行後預期結果

**1. 神片排行榜**

```text
--- 當時 Netflix 評價最高的經典片清單 ---
                                               Name      mean  count
32                   Aqua Teen Hunger Force: Vol. 1  4.168650   6890
67                                      Invader Zim  4.142599   2216
74                            I Love Lucy: Season 2  4.090386   2954
...
```

> 📌 **發現了什麼有趣的現象？**
>
> 排在最上面的不是好萊塢大片，而是**影集、邪典動畫（Cult Animation）、紀錄片**。這反映了 $2005$ 年用 Netflix 的早期使用者特徵——他們是「**找電視看不到的東西**」的小眾族群。

**2. 酸民排行榜**

```text
--- Netflix 史上最嚴格的 5 位評論家 ---
              mean  count
Cust_Id
305344    1.196429    56
1664010   1.250000    52
2118461   1.310345    58
387418    1.320000    50
2439493   1.345455    55
```

> 📌 這 $5$ 位酸民平均給分都不到 $1.5$ 分（滿分 $5$）。換句話說，**他們眼裡幾乎沒有好片**。

---

### ⚠️ 重點觀察：不要只看平均分，要看樣本數

回到神片排行榜，你會發現一個有趣的點：

| 電影 | 平均分 | 評分人數 |
| :--- | :---: | :---: |
| Aqua Teen Hunger Force | $4.17$ | $6{,}890$ |
| The Game（致命遊戲） | $3.85$ | $38{,}362$ |

- 平均分高、樣本少 = **小眾粉絲狂熱**（可能只是死忠粉）
- 平均分中等、樣本多 = **大眾公認的優質作品**

> 💡 **白話翻譯**
>
> 「街角小店 $5$ 顆星（只有 $3$ 個評論）」 vs. 「鼎泰豐 $4$ 顆星（有 $5000$ 個評論）」——你會選哪家？

---

### 🎮 玩玩看

把 `ascending=False` 改成 `ascending=True`，看看「**地雷片排行榜**」：

```python
worst_rated = result[result['count'] > 1000].sort_values(by='mean', ascending=True)
print(worst_rated[['Name', 'mean', 'count']].head(10))
```

你會看到一些「萬人罵」的經典爛片——例如《Fatal Beauty》、《Congo》、《Cube 2: Hypercube》（異次元殺陣 2）。


---


## 🌳 Ch5. 第 4 關：訓練模型（隨機森林）

### 隨機森林是什麼？

跟鐵達尼號用的「神經網路」不同，這次我們要用一個更直觀的模型——**隨機森林（Random Forest）**。

> 💡 **生活比喻：十個朋友幫你判斷**
>
> 想像你要決定要不要看一部電影。你會請 $10$ 個朋友幫你判斷：
> - 朋友 A 看重「**電影本身的評價**」
> - 朋友 B 看重「**這個推薦人平常的眼光**」
> - 朋友 C 看重「**演員陣容**」
> - ⋯⋯
>
> 最後讓 $10$ 個朋友**投票**，得出一個最可能的答案。
> 這就是隨機森林——**隨機**是因為每棵樹看的角度都不一樣，**森林**是因為有很多棵樹。

### 隨機森林 vs. 神經網路

| 比較 | 神經網路 | 隨機森林 |
| :--- | :--- | :--- |
| 學習方式 | 調整數百個旋鈕 | 種一片決策樹 |
| 訓練速度 | 慢 | 快 |
| 解釋性 | 黑盒子 | 可以看每棵樹怎麼判斷 |
| 適合場景 | 影像、語音、文字 | 表格資料 |

對 Netflix 這種**結構化的表格資料**來說，隨機森林又快又準，是業界第一首選。


### 📋 接下來的程式碼會做四件事

1. **製作特徵**：給模型兩個線索——「**電影平均分**」和「**用戶平均分**」。
2. **切分資料**：$80\%$ 訓練、$20\%$ 測試。
3. **訓練模型**：種 $10$ 棵決策樹，每棵最多 $5$ 層。
4. **評估誤差**：用 RMSE（均方根誤差）看模型有多準。

---

#### 🎯 開發歷程：這段程式碼背後的指令（Prompt）

> **指令：訓練隨機森林預測評分**
> 「請用『電影平均分』和『用戶平均分』當作特徵（Feature），『實際評分』當作標籤（Label），訓練一個隨機森林回歸模型（`n_estimators=10, max_depth=5`）。最後印出測試集的 RMSE。」

---

> 🎓 **給學生的學習小提醒**
>
> 這段程式碼裡的 `transform('mean')` 是 Pandas 的進階招式——它會幫你**算出每部電影／每個用戶的平均分，然後填回每一筆資料旁邊**。
>
> 跟之前的 `groupby().mean()` 不同，`transform` 不會把資料聚合成一個小表格，而是**保持原本筆數**，只是新增一欄參考值。


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# === 步驟 1:準備訓練特徵(Features)===
# 線索 1:這部電影的平均分
movie_avg = df.groupby('Movie_Id')['Rating'].transform('mean')
# 線索 2:這個用戶的平均分
user_avg = df.groupby('Cust_Id')['Rating'].transform('mean')

X = pd.DataFrame({
    'movie_avg': movie_avg,
    'user_avg': user_avg
})
y = df['Rating']

# === 步驟 2:切分資料(80% 訓練、20% 測試)===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === 步驟 3:訓練模型 ===
print("模型訓練開始,電腦正在學習規律...")
model = RandomForestRegressor(n_estimators=10, max_depth=5, n_jobs=-1)
model.fit(X_train, y_train)

# === 步驟 4:評估誤差 ===
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"\n訓練完成!模型的誤差 (RMSE) 為:{rmse:.4f}")
print("現在模型已經具備預測能力了。")

模型訓練開始,電腦正在學習規律...

訓練完成!模型的誤差 (RMSE) 為:0.7131
現在模型已經具備預測能力了。


### 📊 執行後預期結果

```text
模型訓練開始，電腦正在學習規律...
訓練完成！模型的誤差（RMSE）為：0.7137
現在模型已經具備預測能力了。
```

> 📌 RMSE 數值會因為每次訓練的隨機性略有差異，但通常會落在 $0.70 \sim 0.75$ 之間。

---

### 🎉 你訓練出了一個 RMSE = $0.7137$ 的模型！

對比歷史數據：

| 模型 | RMSE | 備註 |
| :--- | :---: | :--- |
| Netflix 官方原版系統（Cinematch） | $\sim 0.95$ | $2006$ 年比賽起點 |
| Netflix Prize 冠軍隊伍 | $\sim 0.86$ | $2009$ 年百萬獎金得主 |
| **你的 Notebook 模型** | **$0.71$** | 🎉 |

**等一下⋯⋯這代表你做的模型比百萬美金冠軍還厲害？！**

⚠️ **這就是這份教材最重要的一段——下一關，我們要揭穿這個「假性能」的真相。**


### 🧪 訓練前的小測試：預測「酸民 vs. 好人」對同一部片的反應

在揭穿真相之前，我們先用模型做一個**有趣的對比測試**：

> 同一部電影，**酸民**和**好好先生**會給出多少分？

這就是推薦系統最有價值的地方——它能根據**個人個性**做出**因人而異**的預測。


In [5]:
# 預測函式:輸入用戶平均分 + 電影平均分,預測會給幾顆星
def predict_score(user_avg_score, movie_avg_score):
    test_input = pd.DataFrame([[movie_avg_score, user_avg_score]],
                              columns=['movie_avg', 'user_avg'])
    return model.predict(test_input)[0]

# === 情境測試:一部 4.0 分的電影,在不同人眼中是什麼樣子?===
target_movie_score = 4.0

grumpy_score = predict_score(1.5, target_movie_score)
kind_score = predict_score(4.5, target_movie_score)

print(f"🎬 一部 {target_movie_score} 分的電影,在 AI 眼中:")
print(f"------------------------------------")
print(f"💀 酸民(平均給 1.5 分)會預測給:{grumpy_score:.2f} 星")
print(f"😇 好人(平均給 4.5 分)會預測給:{kind_score:.2f} 星")
print(f"------------------------------------")

# 簡單的邏輯判斷
status = "還可以" if grumpy_score > 2 else "垃圾"
print(f"💡 AI 洞察:這部片在酸民眼裡其實是「{status}」")

🎬 一部 4.0 分的電影,在 AI 眼中:
------------------------------------
💀 酸民(平均給 1.5 分)會預測給:1.71 星
😇 好人(平均給 4.5 分)會預測給:4.56 星
------------------------------------
💡 AI 洞察:這部片在酸民眼裡其實是「垃圾」


### 📊 執行後預期結果

```text
🎬 一部 4.0 分的電影，在 AI 眼中：
------------------------------------
💀 酸民（平均給 1.5 分）會預測給：1.71 星
😇 好人（平均給 4.5 分）會預測給：4.63 星
------------------------------------
💡 AI 洞察：這部片在酸民眼裡其實是「垃圾」
```

> 📌 對酸民來說，$1.71$ 星雖然看起來很低，**但比他平均的 $1.5$ 分要高**——代表這部 $4.0$ 分的電影對他而言，**已經算「相對及格」**。

---

### 🤖 模型學到了什麼？

模型沒有「個性」，但它學會了一條規律：

> **預測分數 ≈ 電影本身的好壞 + 該用戶的個人偏見**

這就是推薦系統的核心邏輯——**校準個人標準，做出因人而異的判斷**。

但⋯⋯⚠️ **這個 RMSE $0.71$ 真的代表模型「很厲害」嗎？**

下一關，我們就要進入這份教材**最重要、也最違反直覺**的章節。


---


## ⚠️ Ch6. 第 5 關：陷阱解密！RMSE $0.71$ 是假的！

### 等等，這個成績怪怪的⋯⋯

我們剛剛得到 RMSE = $0.71$，比 Netflix 比賽冠軍（$0.86$）還好。但這合理嗎？

- Netflix 比賽用了**全部 1 億筆資料**，我們只用了 $50$ 萬筆 → 應該更差才對
- Netflix 冠軍隊用了 $200$ 多種演算法疊加，我們只用一個簡單的隨機森林 → 應該更差才對
- Netflix 冠軍隊花了 $3$ 年，我們訓練只花 $30$ 秒 → 應該更差才對

**那為什麼我們的數字反而更好？**

答案只有一個：**我們作弊了**。

> 💡 **生活比喻：考試偷看答案**
>
> 想像有個學生考試考了 $99$ 分，你心想「他是天才」。
>
> 結果發現他的考卷下面壓著一張**標準答案**——他不是天才，他是**作弊**。
>
> ML 領域，這種「考試前不小心把答案放進考題裡」的錯誤，叫做 **資料洩漏（Data Leakage）**。


### 🔍 我們是怎麼作弊的？（技術解析）

仔細看訓練模型那段程式碼：

```python
movie_avg = df.groupby('Movie_Id')['Rating'].transform('mean')
user_avg = df.groupby('Cust_Id')['Rating'].transform('mean')
```

這兩個特徵——`movie_avg` 和 `user_avg`——是從「**全部 $50$ 萬筆資料的 Rating**」算出來的。

問題是：**這 $50$ 萬筆裡面就包括了測試集那 $20\%$ 的資料**！

也就是說：

> 模型在「猜」一筆評分時，它的特徵裡**已經偷偷包含了這筆評分本身的影響**。

> 💡 **更白話的比喻**
>
> 這就像考試時，你猜「題目答案是 A 還是 B」，
> 但你的「線索」是「全班（包含你自己）選擇 A 的比例」——
> **你把自己的答案也算進線索裡了！**

---

### 🤔 為什麼這個錯誤特別容易發生？

因為它**太自然了**。你在 Ch4 EDA 階段就計算過電影平均分、用戶平均分——那是**合法的探索分析**。但如果你**直接拿這些統計量去訓練模型**，就會出事——因為「平均」這個動作本身，就把測試集的答案混進來了。

這個錯誤每年讓**無數資料科學家**翻車，包括：

- Kaggle 比賽中翻車的隊伍
- 上市公司發布「準確率 99%」的 AI 模型，實際部署後一塌糊塗
- 學術論文被踢爆「測試集污染」而撤稿


### 💡 怎麼修？

**正確的做法是**：`movie_avg` 和 `user_avg` **只能用訓練集的資料來計算**，不能碰到測試集。

```python
# ❌ 錯的（資料洩漏）：用全部資料算平均
movie_avg = df.groupby('Movie_Id')['Rating'].transform('mean')

# ✅ 對的：先切分，再用訓練集算平均
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 用訓練集算每部電影、每個用戶的平均分
movie_avg_dict = train_df.groupby('Movie_Id')['Rating'].mean().to_dict()
user_avg_dict = train_df.groupby('Cust_Id')['Rating'].mean().to_dict()

# 把這些「乾淨的平均分」對應回訓練集和測試集
train_df['movie_avg'] = train_df['Movie_Id'].map(movie_avg_dict)
train_df['user_avg'] = train_df['Cust_Id'].map(user_avg_dict)
test_df['movie_avg'] = test_df['Movie_Id'].map(movie_avg_dict)
test_df['user_avg'] = test_df['Cust_Id'].map(user_avg_dict)

# 然後再用乾淨的特徵訓練
```

---

### 📊 修正後的 RMSE 大概會是多少？

> 🔮 **預測**：用正確方式訓練後，RMSE 會從 $0.71$ 上升到 $0.95 \sim 1.05$ 之間。

**這不是模型變差了**——是**真正的成績**終於浮現出來。

| 版本 | RMSE | 說明 |
| :--- | :---: | :--- |
| 作弊版（資料洩漏） | $0.71$ | 看起來很神，實際無用 |
| 正確版（乾淨訓練） | $\sim 1.0$ | 跟 Netflix 原系統差不多 |
| Netflix 冠軍隊 | $0.86$ | 用了 $200$ 種演算法疊加 |

正確版的 RMSE = $1.0$ 在當時是合理的水準——畢竟我們只用了**兩個特徵**去預測。

### 📊 執行後面更新後的程式

你會發現，這次跑出來的數字應該會變成 1.0 左右。雖然比 0.71 「大」（代表誤差變大），但這才是 **AI 真正靠實力預測出來的結果喔！**


In [6]:
# === 正確版：先切分資料，再計算特徵 ===
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# 1. 先把資料切成「訓練集」和「測試集」
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 2. 【關鍵】只用「訓練集」來計算電影和用戶的平均分 (不准偷看測試集的答案)
movie_avg_dict = train_df.groupby('Movie_Id')['Rating'].mean().to_dict()
user_avg_dict = train_df.groupby('Cust_Id')['Rating'].mean().to_dict()

# 3. 把算好的平均分填回去
train_df['movie_avg'] = train_df['Movie_Id'].map(movie_avg_dict)
train_df['user_avg'] = train_df['Cust_Id'].map(user_avg_dict)
test_df['movie_avg'] = test_df['Movie_Id'].map(movie_avg_dict)
test_df['user_avg'] = test_df['Cust_Id'].map(user_avg_dict)

# 4. 處理測試集中可能出現的新用戶或新電影 (若沒資料就補全體平均)
global_mean = train_df['Rating'].mean()
test_df['movie_avg'] = test_df['movie_avg'].fillna(global_mean)
test_df['user_avg'] = test_df['user_avg'].fillna(global_mean)

# 5. 訓練並預測
X_train = train_df[['movie_avg', 'user_avg']]
y_train = train_df['Rating']
X_test = test_df[['movie_avg', 'user_avg']]
y_test = test_df['Rating']

model_clean = RandomForestRegressor(n_estimators=10, max_depth=5, n_jobs=-1)
model_clean.fit(X_train, y_train)

y_pred_clean = model_clean.predict(X_test)
rmse_clean = root_mean_squared_error(y_test, y_pred_clean)

print(f"修正作弊後的真實 RMSE 為: {rmse_clean:.4f}")

修正作弊後的真實 RMSE 為: 1.1357


### 🎯 終極學習重點：看穿假性能優異模型的能力

這段教材的重點**不是教你修正資料洩漏**（那是進階話題），而是讓你學會一件事：

> ⚠️ **任何看起來「好得不像話」的成績，都應該先懷疑是不是有資料洩漏。**

接下來如果你看到：

- 「我訓練的 AI 預測股價準確率 $99\%$！」
- 「這個醫療影像 AI 比醫生還準！」
- 「Kaggle 比賽我拿了第一名，RMSE $0.001$！」

你應該本能地問：**「特徵裡是不是混了答案？」**

這是**自學 ML 與專業 ML 工程師的關鍵分界線**——專業的人知道「太好的成績通常是假的」，業餘的人則會興奮地分享自己作弊出來的結果。

---

### 🎮 課後挑戰

如果你有時間，可以嘗試：

1. **驗證錯誤**：把 `train_test_split` 改成不同的 `random_state`，看看 RMSE 還是不是 $0.71$ 左右——如果**不論怎麼切都很穩定**，代表特徵真的太強（因為它有偷看答案）。
2. **重訓正確版**：照前面提到的「正確做法」重新訓練一次，觀察 RMSE 變化。
3. **思考其他可能的洩漏**：除了 `movie_avg` 和 `user_avg`，還有什麼特徵也可能有洩漏問題？（💡 提示：評分日期、累積評分次數⋯⋯）


---


## 🛠️ 番外篇：踩坑日記

### 為什麼有番外篇？

這份教材的程式碼在 Kaggle 上「**應該**」會順利執行——但「應該」這兩個字，就是 ML 工程師最害怕的詞。

實際開發時，你會遇到各種**主線教材不會教**的環境地雷。這個番外篇記錄了我自己在 Kaggle 玩這份資料時遇到的 $3$ 個典型陷阱，以及對應的繞道方法。

> 💡 **為什麼要學「踩坑」？**
>
> 大部分 ML 教材都假設「環境完美無瑕」——但真實世界從來不是這樣。
>
> **學會看懂錯誤訊息、知道怎麼繞道，才是真正的工程師能力。**


### 🕳️ 坑 1：NumPy 與 scikit-surprise 版本衝突

#### 錯誤訊息（節錄）

```text
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash...

ImportError: numpy.core.multiarray failed to import
```

#### 為什麼會發生？

- `scikit-surprise` 是專門做推薦系統的套件，是用舊版 NumPy 1.x 編譯的
- Kaggle 預設環境已經升到 NumPy 2.x
- 兩邊不相容，直接 import 就炸

#### 解決方法

```bash
!pip install "numpy<2"
!pip install --force-reinstall scikit-surprise
```

接著**重新啟動 Kernel**（Run → Restart Session），才能讓新版本生效。

> 🎓 **學習啟示**
>
> 真實開發中，**版本衝突**是最常見的痛點。解法通常有三種：
> 1. 降級套件
> 2. 升級套件
> 3. 放棄該套件，改用替代品


### 🕳️ 坑 2：Kaggle 預設關閉 Internet

#### 錯誤訊息

```text
WARNING: Failed to establish a new connection:
[Errno -3] Temporary failure in name resolution
```

#### 為什麼會發生？

- Kaggle Notebook 為了**安全考量**，免費版預設**關閉網路存取**
- 所以你連 `pip install` 都會失敗

#### 解決方法

1. 打開 Notebook 右側 Settings 面板
2. 找到 **Internet** 開關，切換成 **On**
3. Kaggle 會要求**手機驗證**（免費，但需要綁定手機號碼）

#### 不想驗證手機怎麼辦？

放棄 `scikit-surprise`，改用 Kaggle 已經內建的 `scikit-learn`（就是我們本教材主線採用的方案）。

> 🎓 **學習啟示**
>
> **永遠優先使用環境已內建的套件**——這能省下大量除錯時間。
>
> 本教材主線之所以用 `scikit-learn` 的 RandomForest 而非 `scikit-surprise` 的 SVD，就是基於這個原則。


### 🕳️ 坑 3：`mean_squared_error` 的 `squared` 參數消失

#### 錯誤訊息

```text
TypeError: got an unexpected keyword argument 'squared'
```

#### 為什麼會發生？

- 舊版 `sklearn`：用 `mean_squared_error(y_test, y_pred, squared=False)` 取得 RMSE
- 新版 `sklearn`：`squared` 參數被移除，改成獨立函式 `root_mean_squared_error`

#### 解決方法

```python
# ❌ 舊寫法（會報錯）
from sklearn.metrics import mean_squared_error
rmse = mean_squared_error(y_test, y_pred, squared=False)

# ✅ 新寫法（本教材採用）
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_test, y_pred)
```

> 🎓 **學習啟示**
>
> **API 是會變的**——這是套件版本演進中最讓人惱火的事。
>
> 看到 `unexpected keyword argument` 這種錯誤時，先查官方文件看 API 是不是被改了。


---


## 🎓 恭喜你完成了第二個機器學習專案！

在今天的挑戰中，你用跟鐵達尼號完全不同的視角學會了：

| 你學會的核心技能 | 對應章節 |
| :--- | :--- |
| **半結構化資料**（看懂亂格式 .txt 的層次結構） | **Ch2** |
| **資料救援**（用 `ffill` 把消失的 ID 補回來） | **Ch3** |
| **EDA 探索分析**（找神片、酸民，用樣本數過濾偏誤） | **Ch4** |
| **隨機森林**（理解「十個朋友投票」的原理） | **Ch5** |
| ⚠️ **資料洩漏陷阱**（看穿假性能優異的模型） | **Ch6** |
| 🛠️ **真實環境地雷**（版本衝突、網路限制、API 變動） | **番外篇** |

---

### 🚀 你已經跨過了一個關鍵分水嶺

從「**只會跟著教學跑**」到「**會質疑成績、會除錯、會看穿陷阱**」，
這就是 **AI 業餘玩家** 與 **AI 工程師** 之間的最大差別。

很多自學者會卡在「跑得出 RMSE = $0.71$ 就以為自己學會了」這一關。但你不會——你**親眼看到了陷阱**，並且**知道怎麼識別它**。

---

### 💭 給法律人 / 跨領域學員的補充思考

這份教材的「資料洩漏」陷阱，在法律 AI 領域**幾乎天天上演**：

- **AI 預測判決**：用「判決日期之後」才能取得的資料當特徵 → 資料洩漏
- **AI 量刑系統**：用「上訴後的最終結果」回頭算特徵 → 資料洩漏
- **AI 律師選任**：用「結案後」才出現的成功率指標當特徵 → 資料洩漏

> 💡 **核心原則**
>
> **任何「在預測時間點還不存在的資訊」，都不能拿來當特徵。**
>
> 這個原則在法律、金融、醫療領域比性能本身**更重要**——因為一旦發現資料洩漏，整個模型在現實中就**完全無用**。

---

### 🎬 鐵達尼號 + Netflix Prize ML 入門系列正式完結！

**休息一下，期待下次再見！** 🎉
